## Set Environment

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI
openai_client = OpenAI()

## RAG

### LLM Demo 

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

In [4]:
llm('hey whats up')

'Hey! Not much — just here and ready to help. What’s up with you?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Maybe — it depends on the course rules and how far along it is.

In general, you can join now if:
- enrollment is still open,
- the course is self-paced or allows late entry,
- you can catch up on missed material.

If it’s an instructor-led course, you may need:
- permission from the instructor,
- access to recordings or notes,
- a plan to make up missed deadlines.

If you want, I can help you write a short message to the instructor asking if late enrollment is possible.


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants 
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
print(prompt)


Your task is to answer questions from the course participants 
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partic

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still open.


### Set Context Dataset

In [10]:
import requests

In [11]:
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [12]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [13]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

### Set Search Engine

In [14]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [15]:
search_results = index.search(
    question, 
    boost_dict={question: 2.0},
    filter_dict={'course': 'llm-zoomcamp'}, 
    num_results=5
)

In [16]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {question: 2.0}
    filter_dict = {'course': course}

    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict, 
        num_results=5
    )

In [17]:
search_results = search(question)

In [18]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

### Build the Prompt

In [38]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants 
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [20]:
USER_PROMPT_TEMPLATE = """
Question: 
{question}

Context: 
{context}
"""

In [21]:
def build_context(search_results):
    lines = []

    for result in search_results:
        lines.append(result['section'])
        lines.append(f"Q: {result['question']}")
        lines.append(f"A: {result['answer']}")
        lines.append("") 

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question, 
        context=context
    )

    return prompt.strip()

In [23]:
prompt = build_prompt(question, search_results)

print(prompt)

Question: 
I just discovered the course. Can I join now?

Context: 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your proje

In [24]:
lines = []
for result in search_results:
    lines.append(result['section'])
    lines.append(f"Q: {result['question']}")
    lines.append(f"A: {result['answer']}")
    lines.append("")

print("\n".join(lines).strip())

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

### Apply LLM to Build RAG Pipeline

#### Response object attributes and methods

In [25]:
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

In [26]:
response = openai_client.responses.create(
    model = 'gpt-5.4-mini',
    input = prompt
)

In [27]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01e84b42f603bca8006a2b09215efc81a28f48f53a913864e7",
  "created_at": 1781205281.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.4-mini-2026-03-17",
  "object": "response",
  "output": [
    {
      "id": "msg_01e84b42f603bca8006a2b09219dcc81a2a6b103e47954dcc2",
      "content": [
        {
          "annotations": [],
          "text": "Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": "final_answer"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": 1781205281.0,
  "conversation": null,
  "max_output_tokens": null,
  "max_tool_calls": null,
  "moderat

In [28]:
response.output[0]


ResponseOutputMessage(id='msg_01e84b42f603bca8006a2b09219dcc81a2a6b103e47954dcc2', content=[ResponseOutputText(annotations=[], text='Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [29]:
import json
print(json.dumps(response.model_dump()["output"], indent=2, ensure_ascii=False))


[
  {
    "id": "msg_01e84b42f603bca8006a2b09219dcc81a2a6b103e47954dcc2",
    "content": [
      {
        "annotations": [],
        "text": "Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.",
        "type": "output_text",
        "logprobs": []
      }
    ],
    "role": "assistant",
    "status": "completed",
    "type": "message",
    "phase": "final_answer"
  }
]


In [30]:
import json
print(json.dumps(response.model_dump()["output"][0], indent=2, ensure_ascii=False))


{
  "id": "msg_01e84b42f603bca8006a2b09219dcc81a2a6b103e47954dcc2",
  "content": [
    {
      "annotations": [],
      "text": "Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.",
      "type": "output_text",
      "logprobs": []
    }
  ],
  "role": "assistant",
  "status": "completed",
  "type": "message",
  "phase": "final_answer"
}


In [31]:
response.output[0].content[0].text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [32]:
response.output_text

'Yes — you can still join now.\n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [33]:
response.usage

ResponseUsage(input_tokens=336, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=30, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=366)

#### Message History

In [39]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [41]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

#### Full RAG

In [42]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [43]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, you need to submit your project while submissions are still being accepted.


In [44]:
answer = rag("Can I join the course in the middle?")
print(answer)

Yes, you can join after the course has started. If you want a certificate, though, you need to submit your project while submissions are still being accepted.
